In [ ]:
# Import libraries
import openai
import os
import numpy as np
from dotenv import load_dotenv
from neo4j import GraphDatabase

In [ ]:
# Load environment variables
load_dotenv()

# Neo4j connection settings (set these in your .env file)
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USER = os.getenv("NEO4J_USER", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DB_NAME = os.getenv("NEO4J_DB", "hydrologykg")
EMBEDDING_MODEL = "text-embedding-3-large"


if not PASSWORD:
    raise ValueError("Neo4j password is missing. Set it in your .env file.")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

In [ ]:
# Load API key from .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
#OPENAI_API_KEY = os.getenv("ANDRES_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OpenAI API Key is missing in the .env file.")

# Initialize OpenAI client
client = openai.OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
def get_openai_embedding(text, model=EMBEDDING_MODEL):
    """Call OpenAI to get an embedding for the given text."""
    try:
        response = openai.embeddings.create(
            input=text,
            model=model
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"❌ Error generating embedding: {e}")
        return None


In [ ]:
def query_kg_with_embedding(question):
    """
    Converts the input question into an embedding and retrieves relevant nodes and relationships using cosine similarity.

    Args:
        question (str): The user's question.
    Returns:
        list: A list of tuples (source_node, relationship, target_node, similarity_score).
    """
    question_embedding = np.array(get_openai_embedding(question))

    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session(database=DB_NAME) as session:

            # Retrieve nodes, relationships, and their properties
            query = """
            MATCH (n)-[r]->(m)
            WHERE n.embedding IS NOT NULL AND r.embedding IS NOT NULL
            RETURN n, type(r) AS relationship, m, r.embedding AS rel_embedding, r
            """
            nodes = session.run(query)

            results = []

            #'''
            for record in nodes:
                source_node = record["n"]  # Full node object with properties
                relationship = record["relationship"]
                target_node = record["m"]  # Full node object with properties
                rel_embedding = np.array(record["rel_embedding"])
                rel_properties = record["r"]  # Relationship properties

                # Compute cosine similarity
                similarity = np.dot(rel_embedding, question_embedding) / (
                    np.linalg.norm(rel_embedding) * np.linalg.norm(question_embedding)
                )

                # Store the full details
                results.append((source_node, relationship, target_node, similarity, rel_properties))
            #'''
            # Sort by highest similarity score
            results.sort(key=lambda x: x[3], reverse=True)

            return results[:15] if results else []  # Return top 10 relevant relationships



In [ ]:
def generate_rag_response(question, mode="researcher"):
    """
    Retrieves information from the KG and uses it to enrich the LLM's response.
    """
    relevant_data = query_kg_with_embedding(question)
    
    if not relevant_data:
        return "I couldn't find any relevant information in the Knowledge Graph."

    # 🔹 Format the retrieved knowledge with all properties
    retrieved_knowledge = []
    
    for source, rel, target, similarity, rel_props in relevant_data:
        # Convert properties into a readable format
        source_props = ", ".join([f"{key}: {value}" for key, value in source.items() if key != "embedding"])
        target_props = ", ".join([f"{key}: {value}" for key, value in target.items() if key != "embedding"])
        rel_props_text = ", ".join([f"{key}: {value}" for key, value in rel_props.items() if key != "embedding"])

        # Construct relationship representation
        knowledge_text = (
            f"🔹 **{source['name']}** ({source_props}) → *{rel}* ({rel_props_text}) → **{target['name']}** ({target_props}) "
            f"_(Similarity: {similarity:.2f})_"
        )
        retrieved_knowledge.append(knowledge_text)

    # 🔹 Construct the full prompt

    mode_instruction = {
        "researcher": {
            "role": "You are a scientific research assistant helping a hydrology researcher analyze academic insights.",
            "focus": "Highlight theoretical contributions, research gaps, methodologies, and limitations. Be precise and include any numerical evidence, model names, or techniques mentioned."
        },
        "operator": {
            "role": "You are a technical advisor assisting a water utility operator.",
            "focus": "Focus on actionable insights, model applications, practical implications, and decision-making support. Use concrete statistics or recommendations where possible."
        }
    }
    user_prompt_instruction = mode_instruction[mode]["focus"]
    system_role = mode_instruction[mode]["role"]
    
    prompt = f"""
    To respond to the user's question, you are provided with relevant knowledge from the Knowledge Graph.

    **Question:** {question}

    **Retrieved Knowledge from the Knowledge Graph:**
    {'\n'.join(retrieved_knowledge)}

    Based on the retrieved knowledge, generate a concise and accurate natural language response.
    - **Use only the provided knowledge.**
    - **If the data does not fully answer the question, acknowledge it without adding new information.**
    - **{user_prompt_instruction}**
    """

    # print(prompt)  # Debugging: Check the prompt structure

    # 🔹 Generate response with OpenAI GPT-4.1
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": system_role},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )

    return response.choices[0].message.content


In [ ]:
def GPTResponse(prompt, model="gpt-4o"):
    """
    Generates a response using the OpenAI GPT model.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"❌ Error generating GPT response: {e}")
        return None

In [ ]:
# Retreival-augmented generation Testing
questions = [
    "What are the key hydrological challenges specific to the Houston metropolitan area?",
    "What methods have been used to model drought impacts in the Trinity River Basin?",
    "What are the major research gaps in flood risk forecasting around Galveston Bay?"
]

for question in questions:
    print("Question:", question)
    print("\nGPT-4.1")
    print("\nResponse:", GPTResponse(question, model="gpt-4.1"))
    print("\nMode: Researcher")
    print("\nResponse:", generate_rag_response(question, mode="researcher"))
    print("\nMode: Operator")
    print("\nResponse:", generate_rag_response(question, mode="operator"))
    print("\n" + "="*50 + "\n")
